# LeetCode 2996 — Smallest Missing Integer Greater Than Sequential Prefix Sum bằng Python

Link đề: https://leetcode.com/problems/smallest-missing-integer-greater-than-sequential-prefix-sum/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Sorting

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def missingInteger(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm **tiền tố liên tiếp dài nhất** (mỗi số hơn số trước **đúng `1`**), tính **tổng** `S` của nó, rồi tìm **số nguyên nhỏ nhất `x >= S` không có trong `nums`**.
2. Biết cách **duyệt tiền tố và dừng đúng lúc** (tại chỗ đầu tiên không còn "hơn đúng 1"), và hiểu rằng tiền tố chỉ gồm `nums[0]` cũng được coi là liên tiếp.
3. Biết **kiểm tra "có trong mảng"**: `in` với list (`O(n)`) và `in` với `set` (`O(1)` trung bình), và vòng `while x in ...: x += 1` để tìm số "thiếu" đầu tiên từ một mốc.
4. Đi từ cách **list + `while`** → **`set`** → **mảng đánh dấu theo giá trị (vì `nums[i] <= 50`)** → **`itertools.count` + `next`**.
5. Tránh các lỗi kinh điển: kiểm tra `nums[i] > nums[i - 1]` thay vì `== nums[i - 1] + 1`, quên rằng `nums[0]` một mình là tiền tố hợp lệ, tìm số thiếu nhỏ nhất **từ `1`** thay vì từ `S`, dùng `x > S` thay vì `x >= S`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` (đánh chỉ số từ `0`). Một **tiền tố** `nums[0..i]` được gọi là **liên tiếp (sequential)** nếu với mọi `1 <= j <= i` ta có `nums[j] = nums[j - 1] + 1`. Đặc biệt, tiền tố **chỉ gồm `nums[0]`** cũng là tiền tố liên tiếp.

Hãy trả về **số nguyên nhỏ nhất `x`** sao cho:

- `x` **không có mặt trong `nums`**, và
- `x` **lớn hơn hoặc bằng** tổng các phần tử của **tiền tố liên tiếp dài nhất** của `nums`.

**Ví dụ 1**

- Input: `nums = [1, 2, 3, 2, 5]`
- Output: `6`
- Giải thích: tiền tố liên tiếp dài nhất là `[1, 2, 3]` (không mở rộng thêm được vì `2 != 3 + 1`), tổng là `6`. Số `6` không có trong `nums` và `6 >= 6`, nên đáp án là `6`.

**Ví dụ 2**

- Input: `nums = [3, 4, 5, 1, 12, 14, 13]`
- Output: `15`
- Giải thích: tiền tố liên tiếp dài nhất là `[3, 4, 5]`, tổng là `12`. Các số `12`, `13`, `14` đều có trong `nums`, còn `15` thì không, nên đáp án là `15`.

**Ràng buộc:** `1 <= nums.length <= 50`; `1 <= nums[i] <= 50`.

In [1]:
from typing import List
from itertools import count
import random

examples = [
    ([1, 2, 3, 2, 5], 6),
    ([3, 4, 5, 1, 12, 14, 13], 15),
    ([1], 2),
    ([5, 7], 6),
]
for nums, expected in examples:
    print("nums =", nums, "-> kỳ vọng", expected)

nums = [1, 2, 3, 2, 5] -> kỳ vọng 6
nums = [3, 4, 5, 1, 12, 14, 13] -> kỳ vọng 15
nums = [1] -> kỳ vọng 2
nums = [5, 7] -> kỳ vọng 6


## 3. Kiến thức nền

### 3.1. Tiền tố liên tiếp và điều kiện `nums[j] == nums[j - 1] + 1`

"Liên tiếp" nghĩa là **mỗi số hơn số đứng ngay trước đúng `1`**: `3, 4, 5` liên tiếp; `3, 5` thì không (hơn `2`); `3, 3` thì không (hơn `0`); `4, 3` thì không (kém `1`). Chú ý: **`>` không đủ**, phải là **`+ 1` đúng**.

In [2]:
def is_step_one(a, b):
    return b == a + 1

print(is_step_one(3, 4), is_step_one(3, 5), is_step_one(3, 3), is_step_one(4, 3))    # True False False False

True False False False


### 3.2. Tìm độ dài tiền tố rồi cộng: vòng lặp dừng sớm

Ta bắt đầu với tiền tố một phần tử (luôn hợp lệ), rồi mở rộng sang `nums[1]`, `nums[2]`, ... **chừng nào** điều kiện còn đúng. **Dừng ngay** ở chỗ đầu tiên điều kiện sai. Những chỗ liên tiếp **xuất hiện sau đó** không được tính (ví dụ `1, 2, 3, 2, 5, 6`: tiền tố dừng ở `3`, cặp `5, 6` phía sau bị bỏ qua).

In [3]:
nums = [1, 2, 3, 2, 5, 6]
i = 1
while i < len(nums) and nums[i] == nums[i - 1] + 1:
    i += 1
print("độ dài tiền tố:", i, "->", nums[:i], "tổng =", sum(nums[:i]))

độ dài tiền tố: 3 -> [1, 2, 3] tổng = 6


### 3.3. `x in list` và `x in set`

`x in nums` (list) quét từ đầu đến cuối: `O(n)`. Đổi `nums` thành `set` một lần thì `x in s` chỉ tốn `O(1)` trung bình. Với `n <= 50` cả hai đều nhanh, nhưng `set` là thói quen tốt.

In [4]:
nums = [3, 4, 5, 1, 12, 14, 13]
s = set(nums)
print(12 in nums, 15 in nums)            # True False
print(12 in s, 15 in s)                  # True False

True False
True False


### 3.4. Tìm "số thiếu đầu tiên từ một mốc": `while x in s: x += 1`

Bắt đầu từ `x = S`; nếu `x` đang có trong tập thì tăng `x` và thử lại. Vòng lặp **chắc chắn dừng**, vì tập chỉ có hữu hạn phần tử. Khi dừng, `x` là số nhỏ nhất `>= S` không có trong tập.

In [5]:
s = {12, 13, 14, 3, 4, 5, 1}
x = 12
while x in s:
    print("có", x, "-> thử", x + 1)
    x += 1
print("số thiếu đầu tiên từ 12:", x)     # 15

có 12 -> thử 13
có 13 -> thử 14
có 14 -> thử 15
số thiếu đầu tiên từ 12: 15


### 3.5. `itertools.count` và `next` với generator

`count(S)` sinh vô hạn `S, S + 1, S + 2, ...`. `next(x for x in count(S) if x not in s)` lấy **số đầu tiên** thỏa điều kiện rồi dừng (generator lười, không sinh hết). Gọn hơn vòng `while`.

In [6]:
s = {12, 13, 14}
print(next(x for x in count(12) if x not in s))                # 15
print(next(x for x in count(100) if x not in s))               # 100

15
100


### 3.6. Mảng đánh dấu theo giá trị

Vì `nums[i] <= 50`, ta có thể dùng mảng `seen` độ dài `51`: `seen[v] = True` nếu `v` có trong `nums`. Với `x > 50`, chắc chắn `x` **không** có trong `nums` (mọi giá trị `<= 50`), nên không cần tra mảng.

In [7]:
nums = [3, 4, 5, 1, 12, 14, 13]
seen = [False] * 51
for v in nums:
    seen[v] = True
print([v for v in range(51) if seen[v]])      # [1, 3, 4, 5, 12, 13, 14]
x = 12
print(x <= 50 and seen[x])                    # True
print(70 <= 50 and seen[70])                  # False -> 70 không có trong nums

[1, 3, 4, 5, 12, 13, 14]
True
False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Bài gồm **hai bước độc lập**. Với `nums = [3, 4, 5, 1, 12, 14, 13]`:

**Bước 1 — tổng tiền tố liên tiếp.**

| `j` | `nums[j]` | `nums[j - 1] + 1` | khớp? | tiền tố | tổng |
|---|---|---|---|---|---|
| 0 | 3 | (không có) | luôn hợp lệ | `[3]` | 3 |
| 1 | 4 | 4 | có | `[3, 4]` | 7 |
| 2 | 5 | 5 | có | `[3, 4, 5]` | 12 |
| 3 | 1 | 6 | **không** -> dừng | | 12 |

**Bước 2 — số thiếu nhỏ nhất từ mốc `12`.** Thử `12` (có), `13` (có), `14` (có), `15` (**không có**) nên đáp án là `15`.

Mỗi bước đơn giản, bài chỉ là ghép hai bước.

In [8]:
nums = [3, 4, 5, 1, 12, 14, 13]
total = nums[0]
j = 1
while j < len(nums) and nums[j] == nums[j - 1] + 1:
    total += nums[j]
    j += 1
print("tiền tố:", nums[:j], "tổng S =", total)
x = total
while x in nums:
    print(f"x={x} có trong nums -> x += 1")
    x += 1
print("x =", x, "không có trong nums -> đáp án")

tiền tố: [3, 4, 5] tổng S = 12
x=12 có trong nums -> x += 1
x=13 có trong nums -> x += 1
x=14 có trong nums -> x += 1
x = 15 không có trong nums -> đáp án


## 5. Cách 1 — Brute Force / baseline: `list` và `while`

**Ý tưởng.** Tính tổng `S` của tiền tố liên tiếp bằng vòng lặp dừng sớm. Rồi `x = S` và lặp `while x in nums: x += 1` với `nums` là **list** (mỗi lần `in` quét cả list).

In [9]:
def missing_integer_list(nums: List[int]) -> int:
    total = nums[0]                                   # tiền tố một phần tử luôn hợp lệ
    j = 1
    while j < len(nums) and nums[j] == nums[j - 1] + 1:
        total += nums[j]
        j += 1
    x = total
    while x in nums:                                  # x in list: quét O(n)
        x += 1
    return x


for nums, expected in examples:
    print(nums, "->", missing_integer_list(nums), "(kỳ vọng", expected, ")")

[1, 2, 3, 2, 5] -> 6 (kỳ vọng 6 )
[3, 4, 5, 1, 12, 14, 13] -> 15 (kỳ vọng 15 )
[1] -> 2 (kỳ vọng 2 )
[5, 7] -> 6 (kỳ vọng 6 )


In [10]:
# Xem các giá trị x được thử ở ví dụ 2
nums = [3, 4, 5, 1, 12, 14, 13]
total, j = nums[0], 1
while j < len(nums) and nums[j] == nums[j - 1] + 1:
    total += nums[j]
    j += 1
x = total
tried = []
while x in nums:
    tried.append(x)
    x += 1
print("tổng S =", total, "| đã thử (đều có trong nums):", tried, "| dừng ở", x)

tổng S = 12 | đã thử (đều có trong nums): [12, 13, 14] | dừng ở 15


**Độ phức tạp (Cách 1).** Gọi `n = len(nums)`. Tính tổng `O(n)`. Vòng `while x in nums` chạy tối đa `n` lần (vì `x` tăng và `nums` chỉ có `n` số khác nhau), mỗi lần `in` tốn `O(n)`: **Time `O(n²)`**, **Space `O(1)`**.

**Nút thắt.** `x in list` quét lại mảng mỗi lần. Biến `nums` thành `set` một lần thì mỗi lần kiểm tra `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

Giữ nguyên hai bước, chỉ **tăng tốc bước 2**: bước "số `x` có trong `nums` không?" lặp lại nhiều lần với **cùng một tập giá trị**, nên đáng để dựng **cấu trúc tra cứu nhanh** một lần:

1. `seen = set(nums)` (hoặc mảng đánh dấu vì giá trị `<= 50`);
2. `x = S`, tăng `x` đến khi `x not in seen`.

Hơn nữa vì mọi `nums[i] <= 50` nên **với `x > 50` là chắc chắn không có trong `nums`**: vòng `while` luôn kết thúc nhanh (không quá `max(0, 51 - S) + 1` bước, tối đa `51` bước).

## 7. Các cách tiếp theo

### Cách 2 — Dùng `set` cho bước tìm số thiếu

**Giải thích.** Giống Cách 1 nhưng `seen = set(nums)` rồi `while x in seen`. Mỗi lần kiểm tra `O(1)` trung bình.

In [11]:
def missing_integer_set(nums: List[int]) -> int:
    total = nums[0]
    j = 1
    while j < len(nums) and nums[j] == nums[j - 1] + 1:
        total += nums[j]
        j += 1
    seen = set(nums)
    x = total
    while x in seen:                                  # in set: O(1)
        x += 1
    return x


for nums, expected in examples:
    print(nums, "->", missing_integer_set(nums), "(kỳ vọng", expected, ")")

[1, 2, 3, 2, 5] -> 6 (kỳ vọng 6 )
[3, 4, 5, 1, 12, 14, 13] -> 15 (kỳ vọng 15 )
[1] -> 2 (kỳ vọng 2 )
[5, 7] -> 6 (kỳ vọng 6 )


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(n)`** cho `set`.

### Cách 3 — Mảng đánh dấu theo giá trị (vì `1 <= nums[i] <= 50`)

**Giải thích.** `seen = [False] * 51`; đánh dấu mọi giá trị. Với `x` (bắt đầu từ `S`): nếu `x > 50` thì chắc chắn thiếu; ngược lại tra `seen[x]`. Không cần hàm băm, thời gian tra cứu `O(1)` thật sự.

In [12]:
def missing_integer_array(nums: List[int]) -> int:
    total = nums[0]
    j = 1
    while j < len(nums) and nums[j] == nums[j - 1] + 1:
        total += nums[j]
        j += 1
    seen = [False] * 51                               # chỉ số 0..50
    for v in nums:
        seen[v] = True
    x = total
    while x <= 50 and seen[x]:                        # x > 50 chắc chắn không có trong nums
        x += 1
    return x


for nums, expected in examples:
    print(nums, "->", missing_integer_array(nums), "(kỳ vọng", expected, ")")

[1, 2, 3, 2, 5] -> 6 (kỳ vọng 6 )
[3, 4, 5, 1, 12, 14, 13] -> 15 (kỳ vọng 15 )
[1] -> 2 (kỳ vọng 2 )
[5, 7] -> 6 (kỳ vọng 6 )


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(V)`** với `V = 51` (hằng số).

### Cách 4 — `itertools.count` + `next` (gọn, kiểu Python)

**Giải thích.** Gộp bước 2 thành một biểu thức: `next(x for x in count(S) if x not in seen)`. Hoàn toàn tương đương vòng `while`, chỉ khác cách viết. Với bước 1, ta có thể dùng `zip(nums, nums[1:])` để dừng ở cặp đầu tiên không khớp.

In [13]:
def missing_integer_count(nums: List[int]) -> int:
    total = nums[0]
    for prev, cur in zip(nums, nums[1:]):             # các cặp (nums[j-1], nums[j])
        if cur != prev + 1:
            break                                     # dừng ở cặp đầu tiên không liên tiếp
        total += cur
    seen = set(nums)
    return next(x for x in count(total) if x not in seen)


for nums, expected in examples:
    print(nums, "->", missing_integer_count(nums), "(kỳ vọng", expected, ")")

# Quan sát các cặp liên tiếp của ví dụ 2
nums = [3, 4, 5, 1, 12, 14, 13]
print(list(zip(nums, nums[1:])))

[1, 2, 3, 2, 5] -> 6 (kỳ vọng 6 )
[3, 4, 5, 1, 12, 14, 13] -> 15 (kỳ vọng 15 )
[1] -> 2 (kỳ vọng 2 )
[5, 7] -> 6 (kỳ vọng 6 )
[(3, 4), (4, 5), (5, 1), (1, 12), (12, 14), (14, 13)]


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`**.

In [14]:
# Code nộp LeetCode cho Cách 2 (rõ ràng: hai bước tách bạch)
from typing import List

class Solution:
    def missingInteger(self, nums: List[int]) -> int:
        total = nums[0]
        j = 1
        while j < len(nums) and nums[j] == nums[j - 1] + 1:
            total += nums[j]
            j += 1
        seen = set(nums)
        while total in seen:
            total += 1
        return total

## 8. TRACE từng bước

Ta theo dõi Cách 2 với `[3, 4, 5, 1, 12, 14, 13]` (bước 1 rồi bước 2), và trường hợp có cặp liên tiếp **sau** chỗ dừng.

In [15]:
def trace_missing(nums):
    print("nums =", nums)
    total, j = nums[0], 1
    print(f"bước 1: bắt đầu total = nums[0] = {total}")
    while j < len(nums):
        ok = nums[j] == nums[j - 1] + 1
        print(f"  j={j}: nums[j]={nums[j]}, nums[j-1]+1={nums[j - 1] + 1} -> {'khớp: cộng' if ok else 'KHÔNG khớp: dừng'}")
        if not ok:
            break
        total += nums[j]
        j += 1
    print(f"  tiền tố = {nums[:j]}, tổng S = {total}")
    seen = set(nums)
    x = total
    print("bước 2: tìm số nhỏ nhất >= S không có trong nums")
    while x in seen:
        print(f"  x={x} có trong nums -> tăng")
        x += 1
    print(f"  x={x} không có -> đáp án")
    return x


assert trace_missing([3, 4, 5, 1, 12, 14, 13]) == 15
assert trace_missing([1, 2, 3, 2, 5, 6]) == 7          # S = 6 có mặt (số 6 ở cuối) nên 7; cặp 5, 6 phía sau không được cộng

nums = [3, 4, 5, 1, 12, 14, 13]
bước 1: bắt đầu total = nums[0] = 3
  j=1: nums[j]=4, nums[j-1]+1=4 -> khớp: cộng
  j=2: nums[j]=5, nums[j-1]+1=5 -> khớp: cộng
  j=3: nums[j]=1, nums[j-1]+1=6 -> KHÔNG khớp: dừng
  tiền tố = [3, 4, 5], tổng S = 12
bước 2: tìm số nhỏ nhất >= S không có trong nums
  x=12 có trong nums -> tăng
  x=13 có trong nums -> tăng
  x=14 có trong nums -> tăng
  x=15 không có -> đáp án
nums = [1, 2, 3, 2, 5, 6]
bước 1: bắt đầu total = nums[0] = 1
  j=1: nums[j]=2, nums[j-1]+1=2 -> khớp: cộng
  j=2: nums[j]=3, nums[j-1]+1=3 -> khớp: cộng
  j=3: nums[j]=2, nums[j-1]+1=4 -> KHÔNG khớp: dừng
  tiền tố = [1, 2, 3], tổng S = 6
bước 2: tìm số nhỏ nhất >= S không có trong nums
  x=6 có trong nums -> tăng
  x=7 không có -> đáp án


Bảng tóm tắt (Ví dụ 2: `[3, 4, 5, 1, 12, 14, 13]`):

| giai đoạn | giá trị |
|---|---|
| tiền tố liên tiếp dài nhất | `[3, 4, 5]` |
| tổng `S` | `12` |
| `12 in nums`? | có -> `13` |
| `13 in nums`? | có -> `14` |
| `14 in nums`? | có -> `15` |
| `15 in nums`? | **không** -> đáp án `15` |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (vòng `while` + `set`). Đúng class `Solution`, đúng method `missingInteger(self, nums: List[int]) -> int`.

In [16]:
from typing import List

class Solution:
    def missingInteger(self, nums: List[int]) -> int:
        total = nums[0]                                       # tiền tố [nums[0]] luôn hợp lệ
        j = 1
        while j < len(nums) and nums[j] == nums[j - 1] + 1:   # mỗi số hơn số trước đúng 1
            total += nums[j]
            j += 1
        seen = set(nums)
        x = total                                             # tìm số nhỏ nhất >= total không có trong nums
        while x in seen:
            x += 1
        return x


sol = Solution()
for nums, expected in examples:
    print(nums, "->", sol.missingInteger(nums))

[1, 2, 3, 2, 5] -> 6
[3, 4, 5, 1, 12, 14, 13] -> 15
[1] -> 2
[5, 7] -> 6


## 10. Vì sao thuật toán đúng

- **Bước 1 đúng.** Tiền tố liên tiếp là tiền tố mà mọi cặp liền kề thỏa `nums[j] = nums[j - 1] + 1`. Vì tiền tố dài hơn chứa tiền tố ngắn hơn, tập các độ dài hợp lệ là một đoạn `1..L`; ta duyệt `j = 1, 2, ...` và **dừng ở lần vi phạm đầu tiên** nên thu được đúng `L` lớn nhất. Cộng `nums[0..L-1]` được `S`.
- **Bước 2 đúng.** Ta cần `x` nhỏ nhất thỏa `x >= S` và `x not in nums`. Thử `x = S, S + 1, ...` theo thứ tự tăng dần: số đầu tiên không có trong `nums` chính là nhỏ nhất. Vòng lặp dừng vì `nums` hữu hạn.
- **Giới hạn.** Vì mọi `nums[i] <= 50`, mọi `x >= 51` đều không có trong `nums`, nên `x` dừng muộn nhất tại `max(S, 51)`.
- **Hai bước không phụ thuộc nhau** ngoài việc `S` làm mốc xuất phát; mỗi bước dùng đúng định nghĩa nên không có tình huống đặc biệt nào bị bỏ sót (một phần tử, không có số thiếu từ `S` đến 50, ...).
- **Các cặp liên tiếp phía sau không ảnh hưởng.** Đề chỉ xét **tiền tố**, nên các đoạn liên tiếp xuất hiện sau chỗ dừng đầu tiên không được cộng.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `V = 51` là miền giá trị.

- **Cách 1 (`list` + `while`):** Time `O(n²)`, Space `O(1)`.
- **Cách 2 (`set`):** Time `O(n)`, Space `O(n)`.
- **Cách 3 (mảng đánh dấu):** Time `O(n)`, Space `O(V)`.
- **Cách 4 (`count` + `next`):** Time `O(n)`, Space `O(n)`.
- Phải đọc các phần tử nên `O(n)` là tối ưu. Ở đây `n <= 50`, mọi cách chạy tức thì.

## 12. Bộ test

In [17]:
sol = Solution()
impls = {
    "list": missing_integer_list,
    "set": missing_integer_set,
    "array": missing_integer_array,
    "count": missing_integer_count,
    "submit": sol.missingInteger,
}

for nums, expected in examples:
    for key, fn in impls.items():
        got = fn(list(nums))
        assert got == expected, (key, nums, got, expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [18]:
def check_all(nums, expected, label):
    for key, fn in impls.items():
        got = fn(list(nums))
        assert got == expected, (label, key, got, expected)
    print("PASS", label, "->", expected)


check_all([1], 2, "một phần tử: S = 1, có trong nums nên 2")
check_all([50], 51, "một phần tử lớn nhất: S = 50 có mặt, đáp án 51")
check_all([7], 8, "một phần tử 7: 8")
check_all([5, 7], 6, "tiền tố chỉ [5]: S = 5 có mặt nên đáp án 6")
check_all([2, 2, 2], 3, "không có cặp liên tiếp: S = 2 có mặt, 3 thì không")
check_all([1, 2, 3, 4, 5], 15, "toàn mảng liên tiếp: S = 15 không có mặt")
check_all([10, 11, 12], 33, "S = 33 vượt xa giá trị trong mảng")
check_all([1, 2, 3, 3, 4], 6, "tiền tố [1,2,3] dừng ở 3, S = 6; các số sau không ảnh hưởng")
check_all([1, 2, 3, 2, 5, 6], 7, "cặp 5, 6 phía sau chỗ dừng không được cộng: S = 6 có mặt nên 7")
check_all(list(range(1, 51)), 1275, "n = 50, 1..50 liên tiếp: S = 1275 > 50")
check_all([30, 31, 32, 33, 34], 160, "S = 160, vượt miền giá trị")
check_all([4, 5, 6, 15, 16, 17, 18, 19], 20, "S = 15 có mặt, 16..19 có mặt, 20 thiếu")

PASS một phần tử: S = 1, có trong nums nên 2 -> 2
PASS một phần tử lớn nhất: S = 50 có mặt, đáp án 51 -> 51
PASS một phần tử 7: 8 -> 8
PASS tiền tố chỉ [5]: S = 5 có mặt nên đáp án 6 -> 6
PASS không có cặp liên tiếp: S = 2 có mặt, 3 thì không -> 3
PASS toàn mảng liên tiếp: S = 15 không có mặt -> 15
PASS S = 33 vượt xa giá trị trong mảng -> 33
PASS tiền tố [1,2,3] dừng ở 3, S = 6; các số sau không ảnh hưởng -> 6
PASS cặp 5, 6 phía sau chỗ dừng không được cộng: S = 6 có mặt nên 7 -> 7
PASS n = 50, 1..50 liên tiếp: S = 1275 > 50 -> 1275
PASS S = 160, vượt miền giá trị -> 160
PASS S = 15 có mặt, 16..19 có mặt, 20 thiếu -> 20


### 12.2. Tính chất và phản ví dụ

In [19]:
# Tính chất: đáp án >= S >= nums[0], không có trong nums, và mọi số trong [S, đáp án) đều có mặt
def prefix_sum_seq(nums):
    total = nums[0]
    for i in range(1, len(nums)):
        if nums[i] != nums[i - 1] + 1:
            break
        total += nums[i]
    return total

random.seed(11)
for _ in range(1000):
    nums = [random.randint(1, 50) for _ in range(random.randint(1, 50))]
    S = prefix_sum_seq(nums)
    ans = sol.missingInteger(nums)
    assert ans >= S >= nums[0]
    assert ans not in nums
    assert all(v in nums for v in range(S, ans))
print("PASS đáp án >= S, không có trong nums, mọi số từ S đến đáp án đều có mặt")

# Tính chất: nếu nums[1] != nums[0] + 1 thì S = nums[0]
for _ in range(300):
    nums = [random.randint(1, 50) for _ in range(random.randint(2, 50))]
    if nums[1] != nums[0] + 1:
        assert prefix_sum_seq(nums) == nums[0]
print("PASS tiền tố dừng ngay khi nums[1] != nums[0] + 1")

# Tính chất: đầu vào không bị sửa
nums = [3, 4, 5, 1, 12, 14, 13]
copy = list(nums)
for key, fn in impls.items():
    fn(nums)
assert nums == copy
print("PASS đầu vào không bị sửa")

# Phản ví dụ 1: kiểm tra > thay vì + 1
def sai_lon_hon(nums):
    total = nums[0]
    j = 1
    while j < len(nums) and nums[j] > nums[j - 1]:
        total += nums[j]
        j += 1
    seen = set(nums)
    x = total
    while x in seen:
        x += 1
    return x

assert sai_lon_hon([1, 3, 5]) != sol.missingInteger([1, 3, 5])
print("PASS phản ví dụ: '>' cho", sai_lon_hon([1, 3, 5]), "thay vì", sol.missingInteger([1, 3, 5]), "ở [1, 3, 5]")

# Phản ví dụ 2: tìm số thiếu nhỏ nhất từ 1 (mex)
def sai_tu_mot(nums):
    seen = set(nums)
    x = 1
    while x in seen:
        x += 1
    return x

assert sai_tu_mot([1, 2, 3, 2, 5]) == 4 != 6
print("PASS phản ví dụ: tìm từ 1 cho", sai_tu_mot([1, 2, 3, 2, 5]), "thay vì 6")

PASS đáp án >= S, không có trong nums, mọi số từ S đến đáp án đều có mặt
PASS tiền tố dừng ngay khi nums[1] != nums[0] + 1
PASS đầu vào không bị sửa
PASS phản ví dụ: '>' cho 9 thay vì 2 ở [1, 3, 5]
PASS phản ví dụ: tìm từ 1 cho 4 thay vì 6


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [20]:
random.seed(2996)
for trial in range(3000):
    n = random.randint(1, 50)
    mode = random.choice(["random", "seq_prefix", "small"])
    if mode == "random":
        nums = [random.randint(1, 50) for _ in range(n)]
    elif mode == "small":
        nums = [random.randint(1, 8) for _ in range(n)]
    else:
        start = random.randint(1, 50)
        length = random.randint(1, min(n, 51 - start))
        nums = list(range(start, start + length)) + [random.randint(1, 50) for _ in range(n - length)]
    ref = missing_integer_list(list(nums))
    for key, fn in impls.items():
        assert fn(list(nums)) == ref, (key, nums)
print("PASS 3000 bộ ngẫu nhiên (gồm các tiền tố liên tiếp dài) - mọi cách khớp nhau")

PASS 3000 bộ ngẫu nhiên (gồm các tiền tố liên tiếp dài) - mọi cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Kiểm tra `nums[j] > nums[j - 1]` thay vì hơn đúng `1`

In [21]:
def sai_lon_hon_2(nums):
    total = nums[0]
    j = 1
    while j < len(nums) and nums[j] > nums[j - 1]:       # tăng chứ chưa chắc hơn đúng 1
        total += nums[j]
        j += 1
    seen = set(nums)
    while total in seen:
        total += 1
    return total


print("sai  :", sai_lon_hon_2([1, 3, 5]), "| đúng:", Solution().missingInteger([1, 3, 5]))

sai  : 9 | đúng: 2


**Vì sao sai:** "liên tiếp" đòi `nums[j] == nums[j - 1] + 1`. Với `[1, 3, 5]` chỉ `[1]` là tiền tố liên tiếp (vì `3 != 1 + 1`), `S = 1`, đáp án `2`. Dùng `>` ta cộng cả `3, 5` và được `S = 9`.

### Lỗi 2: Quên rằng `nums[0]` một mình đã là tiền tố hợp lệ

In [22]:
def sai_bat_dau_0(nums):
    total = 0                                            # quên cộng nums[0]
    j = 1
    while j < len(nums) and nums[j] == nums[j - 1] + 1:
        total += nums[j]
        j += 1
    seen = set(nums)
    while total in seen:
        total += 1
    return total


print("sai  :", sai_bat_dau_0([5, 7]), "| đúng:", Solution().missingInteger([5, 7]))

sai  : 0 | đúng: 6


**Vì sao sai:** tiền tố gồm **chỉ `nums[0]`** luôn là tiền tố liên tiếp, nên tổng luôn **bắt đầu từ `nums[0]`** (không phải `0`). Với `[5, 7]`: đúng là `S = 5`, `5` có mặt nên đáp án `6`; code sai cho `S = 0` và đáp án `0`.

### Lỗi 3: Tìm số thiếu nhỏ nhất từ `1` thay vì từ `S`

In [23]:
def sai_mex(nums):
    seen = set(nums)
    x = 1
    while x in seen:
        x += 1
    return x


print("sai  :", sai_mex([1, 2, 3, 2, 5]), "| đúng:", Solution().missingInteger([1, 2, 3, 2, 5]))

sai  : 4 | đúng: 6


**Vì sao sai:** đề yêu cầu `x >= S` (với `S` là tổng tiền tố liên tiếp), không phải số nguyên dương nhỏ nhất vắng mặt. Số thiếu nhỏ nhất từ `1` của `[1, 2, 3, 2, 5]` là `4`, nhưng `4 < S = 6` nên không hợp lệ.

### Lỗi 4: Dùng `x > S` thay vì `x >= S`

In [24]:
def sai_lon_hon_S(nums):
    total = nums[0]
    j = 1
    while j < len(nums) and nums[j] == nums[j - 1] + 1:
        total += nums[j]
        j += 1
    seen = set(nums)
    x = total + 1                                        # bỏ qua chính S
    while x in seen:
        x += 1
    return x


print("sai  :", sai_lon_hon_S([1, 2, 3, 2, 5]), "| đúng:", Solution().missingInteger([1, 2, 3, 2, 5]))

sai  : 7 | đúng: 6


**Vì sao sai:** đề cho phép `x` **bằng** `S` (ví dụ 1 có đáp án đúng bằng tổng `6`). Bắt đầu từ `S + 1` bỏ qua chính `S`. Phải bắt đầu từ `x = S`.

### Lỗi 5: Tiếp tục cộng các cặp liên tiếp phía sau chỗ dừng

In [25]:
def sai_khong_dung(nums):
    total = nums[0]
    for i in range(1, len(nums)):
        if nums[i] == nums[i - 1] + 1:                   # không break khi gặp cặp không khớp
            total += nums[i]
    seen = set(nums)
    while total in seen:
        total += 1
    return total


print("sai  :", sai_khong_dung([1, 2, 3, 2, 5, 6]), "| đúng:", Solution().missingInteger([1, 2, 3, 2, 5, 6]))

sai  : 12 | đúng: 7


**Vì sao sai:** chỉ **tiền tố** (đoạn liên tiếp bắt đầu từ đầu mảng) được tính. Khi gặp cặp không khớp (`3 -> 2`) phải **dừng hẳn**; cặp `5, 6` ở phía sau không còn thuộc tiền tố. Dùng `while` có điều kiện (hoặc `break`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề có hai việc nối tiếp ("tính tổng tiền tố ..." rồi "tìm số nhỏ nhất ≥ ... không có trong ...") → **tách thành hai bước** và kiểm tra từng bước.
- Mẫu tiền tố liên tiếp: `total = nums[0]; j = 1; while j < n and cond(j): total += nums[j]; j += 1`.
- Mẫu "số nhỏ nhất ≥ mốc không có trong tập": `x = mốc; while x in seen: x += 1` (hoặc `next(x for x in count(mốc) if x not in seen)`).
- Phân biệt "**hơn đúng 1**" (`== prev + 1`) với "tăng" (`> prev`) và "không giảm" (`>= prev`).
- Biết miền giá trị (`<= 50`) giúp chắc chắn vòng `while` dừng sớm và cho phép dùng mảng đánh dấu.
- Đọc kỹ chữ "**tiền tố**" (phải bắt đầu từ chỉ số `0`) và "**lớn hơn hoặc bằng**".

## 15. Biến thể của bài

1. Đoạn liên tiếp **dài nhất ở bất kỳ chỗ nào** (không nhất thiết là tiền tố): bài "Longest Consecutive Sequence" (cho mảng không sắp xếp, dùng `set`).
2. Đề tương tự nhưng điều kiện là **`x > S`** (nghiêm ngặt).
3. Tiền tố "**tăng nghiêm ngặt**" thay vì hơn đúng `1`.
4. Trả về **mọi** số thiếu trong đoạn `[S, S + k)`.
5. Hậu tố liên tiếp thay cho tiền tố.

Thử biến thể 2, 3 và 5:

In [26]:
def missing_strictly_greater(nums):
    S = prefix_sum_seq(nums)
    seen = set(nums)
    x = S + 1
    while x in seen:
        x += 1
    return x


def prefix_sum_strictly_increasing(nums):
    total = nums[0]
    for i in range(1, len(nums)):
        if nums[i] <= nums[i - 1]:
            break
        total += nums[i]
    return total


def suffix_sum_sequential(nums):
    total = nums[-1]
    for i in range(len(nums) - 2, -1, -1):
        if nums[i] + 1 != nums[i + 1]:
            break
        total += nums[i]
    return total


print("2.", missing_strictly_greater([1, 2, 3, 2, 5]))                     # 7: bắt đầu từ 7
print("3.", prefix_sum_strictly_increasing([1, 3, 5, 4, 9]))               # 9
print("5.", suffix_sum_sequential([9, 1, 2, 3, 4]))                        # 10
assert missing_strictly_greater([1, 2, 3, 2, 5]) == 7
assert prefix_sum_strictly_increasing([1, 3, 5, 4, 9]) == 9
assert suffix_sum_sequential([9, 1, 2, 3, 4]) == 10

2. 7
3. 9
5. 10


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `while` + `x in list` | tra cứu bằng list | `O(n²)` | `O(1)` | hiểu đề, đáp án chuẩn |
| 2. `while` + `set` | tra cứu `O(1)` | `O(n)` | `O(n)` | **cách nộp bài**, rõ ràng |
| 3. Mảng đánh dấu `[False]*51` | dùng giới hạn giá trị | `O(n)` | `O(V)` | giá trị nhỏ, không cần hàm băm |
| 4. `zip` + `count` + `next` | cách viết gọn kiểu Python | `O(n)` | `O(n)` | code gọn khi đã quen |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `sum([4, 5, 6])`, `3 in {1, 2, 3}`, `next(x for x in count(5) if x not in {5, 6, 8})`, `Solution().missingInteger([4])`, `Solution().missingInteger([2, 3, 4, 7])`.

**Bài 2 (tính tay).** Với `nums = [6, 7, 8, 2, 9, 10, 11, 20]`: tìm tiền tố liên tiếp dài nhất, tính `S`, rồi tìm số nhỏ nhất `>= S` không có trong `nums`.

**Bài 3 (điền TODO).** Hoàn thành `missing_integer_todo(nums)` theo Cách 2.

In [27]:
def missing_integer_todo(nums):
    total = nums[0]
    j = 1
    # TODO 1: while j < len(nums) và nums[j] == nums[j - 1] + 1: cộng nums[j] vào total, tăng j
    seen = set(nums)
    x = total
    # TODO 2: while x in seen: tăng x
    return x

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `missing_integer_array2(nums)` bằng **mảng đánh dấu** (Cách 3) mà không nhìn code mẫu. Kiểm tra với bốn ví dụ trong `examples`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `longest_sequential_prefix_len(nums)`: **độ dài** tiền tố liên tiếp dài nhất. (b) Viết `missing_integers(nums, k)`: trả về `k` số **nhỏ nhất** `>= S` không có trong `nums`. (c) Viết `longest_consecutive_anywhere(nums)`: độ dài của đoạn **số nguyên liên tiếp dài nhất** có thể tạo từ các giá trị của `nums` (không cần đứng cạnh nhau, ví dụ `[100, 4, 200, 1, 3, 2]` cho `4`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `15`, `True`, `7`, `5`, `9`. (Với `[4]`: `S = 4` có mặt nên `5`. Với `[2, 3, 4, 7]`: tiền tố `[2, 3, 4]`, `S = 9`, `9` không có mặt nên `9`.) Bài 3: hai vòng `while` như đã gợi ý trong comment.
- **Mức 2 (vừa):** Bài 2: tiền tố `[6, 7, 8]` (vì `2 != 9`), `S = 21`; `21` không có trong `nums` nên đáp án `21`. Bài 4: `seen = [False] * 51`, vòng `while x <= 50 and seen[x]`.
- **Mức 3 (sâu):** Bài 5(a): đếm `L` đến khi vi phạm, nhớ `L >= 1`. 5(b): vòng `x` từ `S` tăng dần, thêm `x` vào kết quả nếu `x not in seen`, đến khi đủ `k`. 5(c): tạo `s = set(nums)`; chỉ bắt đầu đếm tại các số `v` mà `v - 1 not in s`, rồi đi `v, v + 1, ...` đến khi hết; lấy `max`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert sum([4, 5, 6]) == 15 and (3 in {1, 2, 3}) is True
assert next(x for x in count(5) if x not in {5, 6, 8}) == 7
assert Solution().missingInteger([4]) == 5
assert Solution().missingInteger([2, 3, 4, 7]) == 9

# Bài 2
assert prefix_sum_seq([6, 7, 8, 2, 9, 10, 11, 20]) == 21
assert Solution().missingInteger([6, 7, 8, 2, 9, 10, 11, 20]) == 21

# Bài 3
def missing_integer_todo_ans(nums):
    total = nums[0]
    j = 1
    while j < len(nums) and nums[j] == nums[j - 1] + 1:
        total += nums[j]
        j += 1
    seen = set(nums)
    x = total
    while x in seen:
        x += 1
    return x

for nums, expected in examples:
    assert missing_integer_todo_ans(nums) == expected

# Bài 4
def missing_integer_array2(nums):
    total, j = nums[0], 1
    while j < len(nums) and nums[j] == nums[j - 1] + 1:
        total += nums[j]
        j += 1
    seen = [False] * 51
    for v in nums:
        seen[v] = True
    x = total
    while x <= 50 and seen[x]:
        x += 1
    return x

for nums, expected in examples:
    assert missing_integer_array2(nums) == expected

# Bài 5(a)
def longest_sequential_prefix_len(nums):
    L = 1
    while L < len(nums) and nums[L] == nums[L - 1] + 1:
        L += 1
    return L

assert longest_sequential_prefix_len([1, 2, 3, 2, 5]) == 3
assert longest_sequential_prefix_len([5, 7]) == 1
assert longest_sequential_prefix_len([1, 2, 3, 4]) == 4

# Bài 5(b)
def missing_integers(nums, k):
    seen = set(nums)
    x = prefix_sum_seq(nums)
    res = []
    while len(res) < k:
        if x not in seen:
            res.append(x)
        x += 1
    return res

assert missing_integers([3, 4, 5, 1, 12, 14, 13], 3) == [15, 16, 17]
assert missing_integers([1, 2, 3, 2, 5], 2) == [6, 7]

# Bài 5(c)
def longest_consecutive_anywhere(nums):
    s = set(nums)
    best = 0
    for v in s:
        if v - 1 not in s:                      # v là điểm bắt đầu của một đoạn
            length = 1
            while v + length in s:
                length += 1
            best = max(best, length)
    return best

assert longest_consecutive_anywhere([100, 4, 200, 1, 3, 2]) == 4
assert longest_consecutive_anywhere([5]) == 1
assert longest_consecutive_anywhere([1, 2, 4, 5, 6]) == 3
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (hai bước: tổng tiền tố liên tiếp, rồi số nhỏ nhất `>= S` vắng mặt)

```python
from typing import List

class Solution:
    def missingInteger(self, nums: List[int]) -> int:
        total = nums[0]
        j = 1
        while j < len(nums) and nums[j] == nums[j - 1] + 1:
            total += nums[j]
            j += 1
        seen = set(nums)
        while total in seen:
            total += 1
        return total
```

**Checklist**

- [ ] Điều kiện liên tiếp là `nums[j] == nums[j - 1] + 1` (hơn **đúng 1**), không phải `>`?
- [ ] Tổng bắt đầu từ `nums[0]` (tiền tố một phần tử luôn hợp lệ)?
- [ ] Vòng lặp **dừng** ở cặp đầu tiên không khớp (chỉ xét **tiền tố**)?
- [ ] Số tìm được `>= S` (không phải `> S`), và bắt đầu tìm từ `S` (không phải từ `1`)?
- [ ] Dùng `set` (hoặc mảng đánh dấu) để tra cứu `x in ...`?
- [ ] Đã thử: `n = 1`, mảng liên tiếp hết (`1..50`), tiền tố chỉ một phần tử, `S` vượt `50`, cặp liên tiếp xuất hiện sau chỗ dừng?
- [ ] Skeleton đúng `class Solution` / `def missingInteger(self, nums: List[int]) -> int`?